In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4o")

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from IPython.display import display, Image
from langchain_core.messages.utils import (
    trim_messages, 
    count_tokens_approximately
)

async with MultiServerMCPClient(
    {
        "vision": {
            "command": "python",
            # Make sure to update to the full absolute path to your *.py file
            "args": ["/home/cloudGitLab/mcp-server-vision/mcp-server-vision.py"],
            "env": {
                "VISION_CODE":"False", 
                "VISION_URL_BASE":"http://localhost:5189",
                "VISION_PAGE_SIZE": "100"
                },
            "transport": "stdio",
        }
        
    }
) as client:
    tools = client.get_tools()

    def pre_model_hook(state):
        trimmed_messages = trim_messages(
            state["messages"],
            strategy="last",
            token_counter=count_tokens_approximately,
            max_tokens=3000,
            start_on="human",
            end_on=("human", "tool"),
        )
        # You can return updated messages either under `llm_input_messages` or 
        # `messages` key (see the note below)
        return {"llm_input_messages": trimmed_messages}


    agent = create_react_agent(model, tools, pre_model_hook=pre_model_hook)

    display(Image(agent.get_graph().draw_mermaid_png()))

    sw_response = await agent.ainvoke({"messages": "list transactions and data graphs from ADPR application using Vision tool"})

In [ ]:
tools

In [ ]:
sw_response["messages"]

In [ ]:
from langchain_core.messages import ToolMessage, AIMessage
used_tool_calls = dict()
    
if isinstance(sw_response, dict) and "messages" in sw_response:
    for  m in sw_response["messages"]:
        if isinstance(m, ToolMessage) and m.content and m.tool_call_id:
            tcid = m.tool_call_id
            if tcid in used_tool_calls.keys():
                tcd = used_tool_calls[tcid]
            else:
                tcd = dict()
            tcd["response"] = m.content
            used_tool_calls[tcid] = tcd
        elif isinstance(m, AIMessage) and m.additional_kwargs:
            kw = m.additional_kwargs
            if isinstance(kw, dict) and "tool_calls" in kw:
                for tc in kw["tool_calls"]:
                    tcid = tc["id"]
                    if tcid in used_tool_calls.keys():
                        tcd = used_tool_calls[tcid]
                    else:
                        tcd = dict()
                    tcd["tool_call"] = tc["function"]
                    used_tool_calls[tcid] = tcd


In [ ]:
used_tool_calls

In [ ]:
non_tool_related_messages = []
from langgraph.graph import add_messages

if isinstance(sw_response, dict) and "messages" in sw_response:
    for  m in sw_response["messages"]:
        if isinstance(m, ToolMessage):
            print(f"TOOL - [{m.content}]")
            pass
        elif isinstance(m, AIMessage):
            print(f"AI + [{m.content}] / [{m.content}]")
            add_messages(non_tool_related_messages,m)
        else:
            print(f"ELSE + [{m.content}]")
            add_messages(non_tool_related_messages,m)


In [ ]:
non_tool_related_messages